In [112]:
!pip install evaluate > /dev/null


In [113]:
import os
import torch
import random
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSeq2SeqLM
from datasets import load_dataset
import evaluate

In [114]:
SEED = 5541

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## Step 1

In [115]:
# TODO: Select one pretrained causal LM from the writeup
# Note: The models used in this homework are smaller and perform worse than models like ChatGPT, Gemini, etc.
# You should not expect the generated outputs to be high quality, but you will still be able to compute evaluation metrics and human evaluation scores.
MODEL_NAME = "Qwen/Qwen3-0.6B"

# TODO: You will likely need to configure the model/tokenizer (e.g., padding token)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
model.eval()

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

In [116]:
# TODO: Make up 5 prompts that the causal LM will generate a continuation for (e.g., "Once upon a time")
prompts = [
    "Once upon a time",
    "The future of artificial intelligence",
    "In Taiwan",
    "The best way to learn",
    "To be or not to be"
]

## Step 2

In [117]:
# TODO: Implement each decoding algorithm with appropriate generation parameters and return output IDs.
# Implement the decoding algorithms using model.generate(...). See: https://huggingface.co/docs/transformers/en/generation_strategies

# TODO: Greedy decoding
def generate_greedy(model, tokenizer, input_ids, attention_mask=None, max_new_tokens=80):
    return model.generate(input_ids, attention_mask=attention_mask, max_new_tokens=max_new_tokens, do_sample=False)

# TODO: Beam search decoding
def generate_beam(model, tokenizer, input_ids, attention_mask=None, max_new_tokens=80):
    return model.generate(input_ids, attention_mask=attention_mask, max_new_tokens=max_new_tokens, num_beams=5, early_stopping=True)

# TODO: Top-k sampling decoding
def generate_top_k(model, tokenizer, input_ids, attention_mask=None, max_new_tokens=80):
    return model.generate(input_ids, attention_mask=attention_mask, max_new_tokens=max_new_tokens, do_sample=True, top_k=50)

# TODO: Top-p sampling decoding
def generate_top_p(model, tokenizer, input_ids, attention_mask=None, max_new_tokens=80):
    return model.generate(input_ids, attention_mask=attention_mask, max_new_tokens=max_new_tokens, do_sample=True, top_p=0.95)

In [118]:
def generate_decoding_outputs(model, tokenizer, input_text, max_new_tokens=40):
    # TODO: Tokenize input_text
    inputs = tokenizer(input_text, return_tensors="pt").to(device)
    input_ids = inputs["input_ids"]
    attention_mask = inputs["attention_mask"]
    # TODO: Call each decoding function and store output IDs
    greedy_ids = model.generate(input_ids, attention_mask=attention_mask, max_new_tokens=max_new_tokens, do_sample=False)
    beam_ids = model.generate(input_ids, attention_mask=attention_mask, max_new_tokens=max_new_tokens, num_beams=5, early_stopping=True)
    top_k_ids = model.generate(input_ids, attention_mask=attention_mask, max_new_tokens=max_new_tokens, do_sample=True, top_k=50)
    top_p_ids = model.generate(input_ids, attention_mask=attention_mask, max_new_tokens=max_new_tokens, do_sample=True, top_p=0.95)
    # TODO: Decode output IDs to text
    greedy_text = tokenizer.decode(greedy_ids[0], skip_special_tokens=True)
    beam_text = tokenizer.decode(beam_ids[0], skip_special_tokens=True)
    top_k_text = tokenizer.decode(top_k_ids[0], skip_special_tokens=True)
    top_p_text = tokenizer.decode(top_p_ids[0], skip_special_tokens=True)
    # TODO: Return a dictionary mapping decoding algorithm name to decoded text
    return {
        "greedy": greedy_text,
        "beam": beam_text,
        "top_k": top_k_text,
        "top_p": top_p_text
    }

In [119]:
# TODO: Implement perplexity computation for a generated text. See: https://huggingface.co/docs/transformers/main/en/perplexity
def compute_perplexity(model, tokenizer, text):
    inputs = tokenizer(text, return_tensors="pt").to(device)
    input_ids = inputs["input_ids"]
    with torch.no_grad():
        outputs = model(input_ids, labels=input_ids)
        loss = outputs.loss

    # PPL = e^(loss)
    perplexity = torch.exp(loss).item()

    return perplexity

In [120]:
results_intrinsic = []

for prompt in prompts:
    # TODO: Generate outputs for each decoding method
    outputs = generate_decoding_outputs(model, tokenizer, prompt)
    current_row = {"prompt": prompt}
    # TODO: Compute perplexity for each generated output text
    greedy_text = outputs["greedy"]
    beam_text = outputs["beam"]
    top_k_text = outputs["top_k"]
    top_p_text = outputs["top_p"]
    # TODO: Append prompt, 4 output texts, and 4 perplexity scores
    greedy_perplexity = compute_perplexity(model, tokenizer, greedy_text)
    beam_perplexity = compute_perplexity(model, tokenizer, beam_text)
    top_k_perplexity = compute_perplexity(model, tokenizer, top_k_text)
    top_p_perplexity = compute_perplexity(model, tokenizer, top_p_text)

    results_intrinsic.append({
        "prompt": prompt,
        "greedy_text": greedy_text,
        "greedy_perplexity": greedy_perplexity,
        "beam_text": beam_text,
        "beam_perplexity": beam_perplexity,
        "top_k_text": top_k_text,
        "top_k_perplexity": top_k_perplexity,
        "top_p_text": top_p_text,
        "top_p_perplexity": top_p_perplexity
    })

In [121]:
# Convert intrinsic results to dataframe
df_intrinsic = pd.DataFrame(results_intrinsic)
df_intrinsic.head()

,prompt,greedy_text,greedy_perplexity,beam_text,beam_perplexity,top_k_text,top_k_perplexity,top_p_text,top_p_perplexity
0,Once upon a time,"Once upon a time, there were 3000 people in a ...",3.210226,"Once upon a time, there were 10000000000000000...",2.077372,"Once upon a time, there were 30 people in the ...",4.483727,"Once upon a time, in the 19th century, there w...",4.081415
1,The future of artificial intelligence,The future of artificial intelligence in healt...,2.665065,The future of artificial intelligence (AI) in ...,2.127662,The future of artificial intelligence in the w...,2.873804,The future of artificial intelligence in healt...,2.913277
2,In Taiwan,"In Taiwan, the government has implemented a po...",4.161855,"In Taiwan, what is the name of the government ...",2.784796,"In Taiwan, the number of people living in the ...",3.663839,"In Taiwan, there is a concept of ""solar power""...",6.682905
3,The best way to learn,The best way to learn about the future is to l...,4.227780,"The best way to learn is to practice, and the ...",2.492112,The best way to learn the new language is to p...,4.985672,The best way to learn to use a calculator is t...,5.336820
4,To be or not to be,"To be or not to be, or to be, or to be... is a...",4.975733,To be or not to be is a question of life or de...,2.811264,"To be or not to be, that is the question... an...",5.936471,"To be or not to be, but to be... is the questi...",5.996925


In [122]:
# TODO: Compute average perplexity for each decoding method across all prompts and display results in a table
df_intrinsic_average = pd.DataFrame([
    ["greedy", df_intrinsic["greedy_perplexity"].mean()],
    ["beam", df_intrinsic["beam_perplexity"].mean()],
    ["top_k", df_intrinsic["top_k_perplexity"].mean()],
    ["top_p", df_intrinsic["top_p_perplexity"].mean()]
])
df_intrinsic_average.columns = ["decoding_method", "average_perplexity"]
df_intrinsic_average

,decoding_method,average_perplexity
0,greedy,3.848132
1,beam,2.458641
2,top_k,4.388702
3,top_p,5.002268


In [123]:
# Save intrinsic results
df_intrinsic.to_csv("intrinsic.csv", index=False)

## Step 3

In [124]:
# TODO: Select one option from the writeup and load the dataset
DATASET_NAME = "knkarthick/dialogsum"

# TODO: If you choose the cnn_dailymail dataset, you will need to specify a version (you can pick any)
dataset = load_dataset(DATASET_NAME, split="train")
test_data = dataset.select(range(50))

print(f"Loaded {len(test_data)} samples from {DATASET_NAME}.")

Loaded 50 samples from knkarthick/dialogsum.


In [125]:
# TODO: Load the model for your chosen option from the writeup
MODEL_NAME = "gauravkoradiya/T5-Finetuned-Summarization-DialogueDataset"

# TODO: You will likely need to configure the model/tokenizer (e.g., padding token)
tokenizer_task = AutoTokenizer.from_pretrained(MODEL_NAME)
model_task = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
model_task.eval()

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [126]:
!pip install rouge_score bert_score > /dev/null

In [127]:
# TODO: Choose and load one overlap-based metric and one model-based metric. See: https://github.com/huggingface/evaluate/tree/main/metrics

OVERLAP_METRIC_NAME = "rouge"
MODEL_BASED_METRIC_NAME = "bertscore"

overlap_metric = evaluate.load(OVERLAP_METRIC_NAME)
model_metric = evaluate.load(MODEL_BASED_METRIC_NAME)

In [128]:
# TODO: Implement a function to compute evaluation metrics for each generated output text compared to the reference text.
# If your selected evaluation metric returns multiple scores, only return the most appropriate score for each sample (e.g., RougeL or F1).
# Make sure to justify this selection in your report.
def compute_evaluation_metrics(predictions, references):
    # TODO: Compute overlap metric scores
    rouge_results = overlap_metric.compute(
        predictions=predictions,
        references=references,
        use_aggregator=False
    )
    overlap_scores = rouge_results["rougeL"]
    # TODO: Compute model-based metric scores
    bert_results = model_metric.compute(
        predictions=predictions,
        references=references,
        lang="en",
        model_type="distilbert-base-uncased"
    )

    model_scores = bert_results["f1"]
    # TODO: Return a dictionary mapping evaluation metric to list of scores
    return {
        "overlap": overlap_scores,
        "model": model_scores
    }

## Step 4

In [129]:
results_extrinsic = []

for sample in test_data:
    # TODO: Get the input and reference text for your selected dataset
    input_text = sample["dialogue"]
    reference = sample["summary"]

    # TODO: Generate outputs for the input text
    outputs = generate_decoding_outputs(model_task, tokenizer_task, input_text, max_new_tokens=80 )
    # TODO: Append input text, reference text, and generated output texts
    results_extrinsic.append({
        "input_text": input_text,
        "reference": reference,
        "greedy": outputs["greedy"],
        "beam": outputs["beam"],
        "top_k": outputs["top_k"],
        "top_p": outputs["top_p"]
    })

In [130]:
# Convert extrinsic results to dataframe
df_extrinsic = pd.DataFrame(results_extrinsic)

for method in ["greedy", "beam", "top_k", "top_p"]:
    # TODO: Create lists of predictions and references for the current decoding method
    predictions = df_extrinsic[method].tolist()
    references = df_extrinsic["reference"].tolist()
    # TODO: Compute evaluation metric scores for the current decoding method
    metrics = compute_evaluation_metrics(predictions, references)
    # TODO: Append evaluation metric scores
    df_extrinsic[f"{method}_overlap"] = metrics["overlap"]
    df_extrinsic[f"{method}_model"] = metrics["model"]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [132]:
extrinsic_average = []

for method in ["greedy", "beam", "top_k", "top_p"]:
    # TODO: Compute means of evaluation metric scores for the current decoding method
    overlap_mean = df_extrinsic[f"{method}_overlap"].mean()
    model_mean = df_extrinsic[f"{method}_model"].mean()
    # TODO: Append evaluation metric score means
    extrinsic_average.append({
        "method": method,
        "overlap_mean": overlap_mean,
        "model_mean": model_mean
    })

df_extrinsic_average = pd.DataFrame(extrinsic_average)
df_extrinsic_average

,method,overlap_mean,model_mean
0,greedy,0.293567,0.834340
1,beam,0.284886,0.840216
2,top_k,0.245650,0.826944
3,top_p,0.262112,0.827162


In [133]:
# Save extrinsic results
df_extrinsic.to_csv("extrinsic.csv", index=False)

## Step 5

Note: The following code is provided to simplify human evaluation. However, you will need to manually open the `human.csv` file and score each generated output text (5 prompts * 4 decoding algorithms * 3 human evaluation metrics = 60 manually evaluated scores)

In [134]:
# Create human evaluation template from first 5 rows
df_human = df_extrinsic.head(5).copy()

# Create a template for human evaluation
for method in ["greedy", "beam", "top_k", "top_p"]:
    df_human[f"{method}_factuality"] = pd.NA
    df_human[f"{method}_fluency"] = pd.NA
    df_human[f"{method}_coherence"] = pd.NA

if not os.path.exists("human.csv"):
    df_human.to_csv("human.csv", index=False)

# TODO: Open human.csv and manually give a 1-5 score for each generated output text based on factuality, fluency, and coherence.

In [136]:
# Load completed human annotations
df_human = pd.read_csv("human.csv")

for method in ["greedy", "beam", "top_k", "top_p"]:
    factuality_mean = df_human[f"{method}_factuality"].mean()
    fluency_mean = df_human[f"{method}_fluency"].mean()
    coherence_mean = df_human[f"{method}_coherence"].mean()

    df_extrinsic_average.loc[df_extrinsic_average["method"] == method, "factuality_mean"] = factuality_mean
    df_extrinsic_average.loc[df_extrinsic_average["method"] == method, "fluency_mean"] = fluency_mean
    df_extrinsic_average.loc[df_extrinsic_average["method"] == method, "coherence_mean"] = coherence_mean

df_extrinsic_average

,method,overlap_mean,model_mean,factuality_mean,fluency_mean,coherence_mean
0,greedy,0.293567,0.834340,2.4,1.8,1.8
1,beam,0.284886,0.840216,1.6,2.2,1.8
2,top_k,0.245650,0.826944,1.4,3.0,1.6
3,top_p,0.262112,0.827162,1.6,2.0,1.8
